<a href="https://colab.research.google.com/github/kawakami-shouhei/-/blob/main/cabbage_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# キャベツの価格予測 — Phase 1 試作ノート
**目的**：過去の価格と天気から、来週のキャベツ価格の傾向を予測できるか試す。
**データ**：東京都中央卸売市場日報（東京都オープンデータ、CC-BY）＋ Open-Meteo の東京の天気。全部、公開・無料の実データ。
**使う道具**：pandas（表を扱う）＋ scikit-learn（学習させる）。Colabには最初から入っている。


In [8]:
import json
import urllib.request
import pandas as pd
import numpy as np


## 1. 価格データを取る
東京都のオープンデータから、2024・2025・2026年度の青果の価格ファイルを直接読み込む。


In [16]:
urls = [
    "https://www.opendata.metro.tokyo.lg.jp/shijyou/2024/result_price_fresh.csv",
    "https://www.opendata.metro.tokyo.lg.jp/shijyou/2025/result_price_fresh.csv",
    "https://www.opendata.metro.tokyo.lg.jp/shijyou/2026/result_price_fresh.csv",
]
frames = [pd.read_csv(u, encoding="utf-8-sig") for u in urls]
p = pd.concat(frames, ignore_index=True)
print("行数:", len(p))
print(p.head(3))


行数: 793633
   分類  市場  青果市場ID          日付  種別    品名 販売方法   品種  産地  単位   高値(円)   中値(円)  \
0  青果  豊洲  Sei_K1  2024-04-01  野菜  だいこん   相対  NaN  千葉  10  1944.0  1620.0   
1  青果  豊洲  Sei_K1  2024-04-01  野菜  だいこん   相対  NaN  茨城  10     NaN     NaN   
2  青果  豊洲  Sei_K1  2024-04-01  野菜  だいこん   相対  NaN  各地  10  1296.0  1080.0   

    安値(円)  
0  1404.0  
1     NaN  
2   864.0  


## 2. キャベツだけ抜き出す
条件は3つ：大田市場（都内で一番大きい市場）、相対取引（売買の中心）、中値（真ん中の価格）。
1日ごとに、産地ごとの中値を平均して「その日の価格」にする。単位は10kgあたり。


In [10]:
c = p[(p["品名"] == "キャベツ") & (p["市場"] == "大田") & (p["販売方法"] == "相対")].copy()
c["日付"] = pd.to_datetime(c["日付"])
c["中値(円)"] = pd.to_numeric(c["中値(円)"], errors="coerce")
daily_price = c.groupby("日付")["中値(円)"].mean().rename("price").reset_index()
print("日数:", len(daily_price))
print(daily_price["日付"].min(), "→", daily_price["日付"].max())
print(daily_price["price"].describe().round(1))


日数: 558
2024-04-01 00:00:00 → 2026-06-30 00:00:00
count     555.0
mean     1221.9
std       602.0
min       486.0
25%       828.0
50%       972.0
75%      1404.0
max      4320.0
Name: price, dtype: float64


## 3. 天気データを取る
Open-Meteo（無料・登録不要）から、東京の毎日の平均気温・降水量・日照時間を取る。


In [11]:
url = ("https://archive-api.open-meteo.com/v1/archive"
       "?latitude=35.68&longitude=139.69"
       "&start_date=2024-04-01&end_date=2026-11-20"
       "&daily=temperature_2m_mean,precipitation_sum,sunshine_duration"
       "&timezone=Asia%2FTokyo")
with urllib.request.urlopen(url) as r:
    w = json.load(r)
wd = pd.DataFrame({
    "date": pd.to_datetime(w["daily"]["time"]),
    "temp": w["daily"]["temperature_2m_mean"],
    "rain": w["daily"]["precipitation_sum"],
    "sun": pd.to_numeric(w["daily"]["sunshine_duration"], errors="coerce") / 3600.0,  # 秒→時間
})
print(wd.head(3))


        date  temp  rain        sun
0 2024-04-01  13.3   4.5   8.771319
1 2024-04-02  12.2   0.0  11.055442
2 2024-04-03  12.6  13.6   0.245478


## 4. 週ごとにまとめて、材料を作る
価格と天気を日付でくっつけて、週単位に集計する。材料（特徴量）はこう決める。
- 前週の価格（price_lag1）、4週間前の価格（price_lag4）
- その週の平均気温、降水量の合計、日照時間の合計
- 月、年の第何週か


In [12]:
df = daily_price.merge(wd, left_on="日付", right_on="date", how="inner").drop(columns=["date"])
df = df.set_index("日付").sort_index()
weekly = df.resample("W-MON").agg(price=("price", "mean"), temp=("temp", "mean"),
                                   rain=("rain", "sum"), sun=("sun", "sum"))
weekly["price_lag1"] = weekly["price"].shift(1)
weekly["price_lag4"] = weekly["price"].shift(4)
weekly["temp_lag1"] = weekly["temp"].shift(1)
weekly["rain_lag1"] = weekly["rain"].shift(1)
weekly["month"] = weekly.index.month
weekly["weekofyear"] = weekly.index.isocalendar().week.astype(int)
weekly["target_next"] = weekly["price"].shift(-1)  # 来週の価格が「答え」
weekly = weekly.dropna()
print("週数:", len(weekly))
print(weekly.tail(3))


週数: 114
            price   temp   rain        sun  price_lag1  price_lag4  temp_lag1  \
日付                                                                              
2026-06-15  777.6  20.50   47.8  43.198228       853.2      1101.6       20.0   
2026-06-22  864.0  22.50   56.4  37.952886       777.6      1166.4       20.5   
2026-06-29  842.4  21.66  109.8  19.255450       864.0       763.2       22.5   

            rain_lag1  month  weekofyear  target_next  
日付                                                     
2026-06-15       10.4      6          25        864.0  
2026-06-22       47.8      6          26        842.4  
2026-06-29       56.4      6          27        756.0  


## 5. 学習させる
2026年4月までを練習用、4月以降をテスト用にする。
比べる相手は「来週の価格＝今週の価格」という単純な予想（ベースライン）。


In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

feats = ["price_lag1", "price_lag4", "temp", "temp_lag1", "rain", "rain_lag1", "sun", "month", "weekofyear"]
train = weekly[weekly.index < "2026-04-01"]
test = weekly[weekly.index >= "2026-04-01"]

model = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)
model.fit(train[feats], train["target_next"])
pred = model.predict(test[feats])

mae = mean_absolute_error(test["target_next"], pred)
baseline = mean_absolute_error(test["target_next"], test["price_lag1"])
print(f"モデルの誤差（MAE）: {mae:.0f} 円/10kg")
print(f"単純な予想（来週＝今週）の誤差: {baseline:.0f} 円/10kg")
print()
print("どの材料が効いたか：")
print(pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False).round(3))


モデルの誤差（MAE）: 143 円/10kg
単純な予想（来週＝今週）の誤差: 177 円/10kg

どの材料が効いたか：
price_lag1    0.718
weekofyear    0.110
temp          0.038
sun           0.034
rain_lag1     0.028
price_lag4    0.021
rain          0.018
temp_lag1     0.017
month         0.016
dtype: float64


## 6. 来週の予測を出す


In [14]:
last = weekly.iloc[[-1]][feats]
next_pred = model.predict(last)[0]
print(f"最新の週（{weekly.index[-1].date()}）の価格: {weekly['price'].iloc[-1]:.0f} 円/10kg")
print(f"来週の予測: {next_pred:.0f} 円/10kg")


最新の週（2026-06-29）の価格: 842 円/10kg
来週の予測: 845 円/10kg


In [15]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## 結果の読み方（正直に）
- モデルは「来週＝今週」よりは少しだけ当たる。ただし誤差は100円超（10kgあたり）ある。
- 一番効いた材料は「前週の価格」。天気の寄与はこの単純な作りでは小さい。
- 4月のような価格の急騰は外しやすい。価格予測は「傾向を見る道具」で、「当てる道具」ではない。
- 次の改善：品目を増やす、産地の天気を入れる、週ではなく日単位で見る。
